In [2]:
import pandas as pd 

# Load the data
df = pd.read_excel("../data/raw/10-6-20 Polling places.xls")

# Calculate Total Polling Locations per County
unique_locations = df.drop_duplicates(subset=['COUNTY', 'LOCATION_ADDRESS'])
county_counts = unique_locations.groupby('COUNTY').size().reset_index(name='Polling_Place_Count')
county_counts = county_counts.sort_values(by='Polling_Place_Count', ascending=False)

# Calculate Average Precincts per Polling Location (Resource Strain)
precincts_per_location = df.groupby(['COUNTY', 'LOCATION_ADDRESS'])['PRECINCT'].nunique().reset_index(name='Precincts_Served')
avg_precincts_served = precincts_per_location.groupby('COUNTY')['Precincts_Served'].mean().reset_index(name='Avg_Precincts_per_Location')
avg_precincts_served = avg_precincts_served.sort_values(by='Avg_Precincts_per_Location', ascending=False)

print(avg_precincts_served.head(10))

       COUNTY  Avg_Precincts_per_Location
28       Dade                   12.000000
114     Worth                    8.000000
106  Sullivan                    6.333333
91     Ripley                    6.000000
56      Lewis                    3.833333
43       Holt                    3.750000
40   Harrison                    3.333333
104  Stoddard                    3.333333
2    Atchison                    3.200000
45     Howell                    3.142857


In [8]:
import os
import glob

# Look inside your raw data folder for anything ending in .shp
shapefiles = glob.glob("../data/raw/**/*.shp", recursive=True)

import geopandas as gpd

# 1. Load your Missouri 2020 Precincts Shapefile
print("Loading shapefile...")
precincts_gdf = gpd.read_file("../data/raw/MO 2020 Precincts.shp")

# 2. Automatically repair any tiny geometry glitches in the raw Census file
precincts_gdf['geometry'] = precincts_gdf.geometry.make_valid()

# 3. Project to a universal Equal Area projection (Meters)
precincts_gdf = precincts_gdf.to_crs(epsg=5070)

# 4. Calculate Area (Square Meters to Square Miles)
precincts_gdf['Area_Sq_Miles'] = precincts_gdf.geometry.area * 0.000000386102

print(f"Current CRS for spatial join: {precincts_gdf.crs}")

# 5. Review the largest and smallest precincts
print("\nTop 5 Largest Precincts (by Sq Miles):")
print(precincts_gdf[['NAME20', 'COUNTYFP20', 'Area_Sq_Miles']].sort_values(by='Area_Sq_Miles', ascending=False).head())

Loading shapefile...
Current CRS for spatial join: EPSG:5070

Top 5 Largest Precincts (by Sq Miles):
                 NAME20 COUNTYFP20  Area_Sq_Miles
939            Eminence        203     274.508226
771       Benson Center        083     230.567590
206            Novinger        001     213.247117
4534  Macon Expo Center        121     208.658979
518             Central        123     191.184117


In [9]:
import geopandas as gpd
import pandas as pd

# 1. Load the fully geocoded polling locations
print("Loading geocoded polling locations...")
polling_gdf = gpd.read_file("../data/raw/MO_2020_Polling_Geocoded_Full.geojson")

# 2. MATCH THE PROJECTIONS (Crucial Step!)
# The points must be in the exact same EPSG:5070 projection as the precincts
polling_gdf = polling_gdf.to_crs(epsg=5070)

# 3. THE SPATIAL JOIN
# This matches every polling point to the precinct polygon it physically sits inside
print("Joining points to polygons...")
precincts_with_polls = gpd.sjoin(precincts_gdf, polling_gdf, how="left", predicate="intersects")

# 4. Count the polling locations per precinct
# We group by the precinct's unique ID ('GEOID20') and count the valid polling locations
poll_counts = precincts_with_polls.groupby('GEOID20')['LOCATION_NAME'].count().reset_index(name='Polling_Locations_Count')

# Merge those counts back onto our pristine precinct map
final_precincts_gdf = precincts_gdf.merge(poll_counts, on='GEOID20')

# 5. CALCULATE THE KEY METRIC: DENSITY
# How many polling places exist per square mile in this precinct?
final_precincts_gdf['Polls_Per_Sq_Mile'] = final_precincts_gdf['Polling_Locations_Count'] / final_precincts_gdf['Area_Sq_Miles']

# 6. Find the "Resource Deserts" for your slide deck
# Let's look at the largest precincts geographically that only have 1 (or 0) polling locations
resource_deserts = final_precincts_gdf[final_precincts_gdf['Polling_Locations_Count'] <= 1]

print("\n--- TOP 5 'RESOURCE DESERTS' (Massive area, 1 or 0 polls) ---")
print(resource_deserts[['NAME20', 'COUNTYFP20', 'Area_Sq_Miles', 'Polling_Locations_Count']].sort_values(by='Area_Sq_Miles', ascending=False).head())

# (Optional) Save this ultimate dataset to map in QGIS or Python later!
# final_precincts_gdf.to_file("../data/raw/MO_2020_Precincts_with_Density.geojson", driver="GeoJSON")

Loading geocoded polling locations...


DataSourceError: ../data/raw/MO_2020_Polling_Geocoded_Full.geojson: No such file or directory

In [ ]:
import matplotlib.pyplot as plt

# 1. Set up a large, high-quality canvas for the PowerPoint slide
fig, ax = plt.subplots(1, 1, figsize=(15, 12))
ax.set_title("Missouri 2020: Polling Location Density by Precinct", fontsize=20, pad=20)

# 2. Turn off the map axes (we don't need latitude/longitude numbers on a slide)
ax.axis('off')

# 3. Plot the data!
# We are coloring the map based on the 'Polls_Per_Sq_Mile' column we calculated.
# 'cmap' sets the color scheme (OrRd = Orange to Red).
# 'scheme' groups the data into readable chunks (Quantiles).
final_precincts_gdf.plot(
    column='Polls_Per_Sq_Mile',
    ax=ax,
    cmap='OrRd',
    linewidth=0.2,
    edgecolor='black', # Draws thin borders around the precincts
    legend=True,
    legend_kwds={
        'loc': 'lower right',
        'title': 'Polls per Sq Mile\n(Darker = Higher Density)',
        'fontsize': 12,
        'title_fontsize': 14
    },
    missing_kwds={
        "color": "lightgrey", # Colors any precincts that had 0 polling locations
        "edgecolor": "black",
        "hatch": "///",
        "label": "Missing/Zero Data"
    }
)

# 4. Save the map as a high-res image directly to your folder
plt.savefig("../data/MO_2020_Density_Map.png", dpi=300, bbox_inches='tight')

print("Map generated and saved as 'MO_2020_Density_Map.png'!")
plt.show() # Displays the map right here in the notebook